## 08. Asynchronous Communication Parallel Strategy

UltraOpt???????????????????`HpBandSter`???,?????[HpBandSter??](https://automl.github.io/HpBandSter/build/html/index.html)????????????

---


UltraOpt????3?????:

|????|???|????????|
|-------|-----|-------------|
|??| Serial| No |
|MapReduce| MapReduce| No |
|????|AsyncComm| **Yes** |

UltraOpt???????? `AsyncComm`, ? `????`??????? `n_jobs=1` ?? `multi_fidelity_iter_generator=None` ,UltraOpt??????????? `Serial` 

??????????,???`fmin`???`auto_identify_serial_strategy=False`

---

???????UltraOpt?????????????,????????`ask`(????)?`tell`(????)???????,??????????????(Pyro4)?????????:

![AsyncComm](https://img-blog.csdnimg.cn/20210102124133367.jpg)

??????????,?????????:

- **NameServer** ??????????? | [HpBandSter????](https://automl.github.io/HpBandSter/build/html/core/nameserver.html)
- **Master** ????????????????,??????????????,????????(?AutoML????????)
- **Worker** ???????????????,?Master???, ?????????(?AutoML????????),???????`Dispacher`
- **Dispacher** ???,???`Master`????????`Worker`,???`Worker`???`Master`


---

??????????,???Worker???`ultraopt.fmin`???`Workers`?`Master`

In [ ]:
from ultraopt.async_comm.master import Master
from ultraopt.async_comm.worker import Worker
from ultraopt.async_comm.nameserver import NameServer
from ultraopt.utils.net import get_a_free_port
import logging
logging.basicConfig(level=logging.INFO)  # verbose print

### ?? NameServer

In [ ]:
ns_host = "0.0.0.0"
port = get_a_free_port(9090, ns_host)

In [ ]:
NS = NameServer(run_id="learn_ultraopt", host=ns_host, port=port)

In [ ]:
NS.start()

### ?? Worker

????AutoML???`Worker` (?????,??????????????)

- ??1. ????`Worker`,?`????`?????

In [ ]:
# ??automl????
from ultraopt.tests.automl import evaluator

In [ ]:
worker1 = Worker(
    run_id="learn_ultraopt",
    nameserver=ns_host,
    nameserver_port=port,
    worker_id=0
)

?`????`?????

In [ ]:
worker1.initialize(evaluator) 

??**??**?**??**??Worker

In [ ]:
worker1.run(
    background=True,
    concurrent_type="thread" # ????? process ,?????Python3.7+??????,??????
)

- ??2. ????`Worker`???`Worker`,??`compute`??

??? `05. Implement a Simple AutoML System` ??????, AutoML????? `????` ,??????,`????` ????????????**????**???????`Worker`???????**????**?????

?**????**?,???????????????`Worker`???????????`Worker`?**??**`Worker`,???`compute`???

?????? `06. Combine Multi-Fidelity Optimization` ?? `ultraopt.tests.automl.Evaluator` ????????,?? `AutoMLWorker` ???????3?:

- note 1 : ???????????
- note 2 : ???? `compute` ??
- note 3 : ?? `compute` ???????????,????? `loss` ?

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import sklearn.metrics
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold  
import numpy as np

from ultraopt.hdl import layering_config

default_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

class AutoMLWorker(Worker):
    def __init__(
        self,
        X, y,
        metric="accuracy",
        cv=default_cv,
        **kwargs
    ):
        self.X = X
        self.y = y
        self.metric = metric
        self.cv = cv
        # note 1 : ???????????
        super(AutoMLWorker, self).__init__(**kwargs)
    
    # note 2 : ???? `compute` ??
    def compute(self, config_id, config, config_info, budget, working_directory):
        '''????? compute ??'''
        layered_dict = layering_config(config)
        AS_HP = layered_dict['classifier'].copy()
        AS, HP = AS_HP.popitem()
        ML_model = eval(AS)(**HP)
        sample_ratio  = budget
        scores = []
        for i, (train_ix, valid_ix) in enumerate(self.cv.split(X, y)):
            rng = np.random.RandomState(i)
            size = int(train_ix.size * sample_ratio)
            train_ix = rng.choice(train_ix, size, replace=False)
            X_train = X[train_ix, :]
            y_train = y[train_ix]
            X_valid = X[valid_ix, :]
            y_valid = y[valid_ix]
            ML_model.fit(X_train, y_train)
            y_pred = ML_model.predict(X_valid)
            score = eval(f"sklearn.metrics.{self.metric}_score")(y_valid, y_pred)
            scores.append(score)
        score = np.mean(scores)
        # note 3 : ?? `compute` ???????????,????? `loss` ?
        return {
            "loss": 1 - score  
        }

????????????????????worker

In [ ]:
X, y = load_digits(return_X_y=True)
worker2 = AutoMLWorker(
    X, y, 
    # ??????????????
    run_id="learn_ultraopt",
    nameserver=ns_host,
    nameserver_port=port,
    worker_id=1
)

In [ ]:
worker2.run(background=True)

### ?? Master

In [ ]:
from ultraopt.optimizer import ETPEOptimizer
from ultraopt.multi_fidelity import HyperBandIterGenerator
from ultraopt.utils.progress import default_callback
from ultraopt.tests.automl import config_space
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ??tabular_nn ????
logging.getLogger("tabular_nn.component.embedding_encoder.EmbeddingEncoder").setLevel(logging.WARNING)
logging.getLogger("tabular_nn.entity_embedding_nn.TrainEntityEmbeddingNN").setLevel(logging.WARNING)

In [ ]:
optimizer = ETPEOptimizer()

In [ ]:
iter_generator = HyperBandIterGenerator(min_budget=1/4, max_budget=1, eta=2)

In [ ]:
optimizer.initialize(config_space=config_space, budgets=iter_generator.get_budgets())

In [ ]:
master = Master(
    run_id="learn_ultraopt", 
    optimizer=optimizer, 
    iter_generator=iter_generator, 
    progress_callback=default_callback,
    checkpoint_file=None, # ??????
    checkpoint_freq=1,
    nameserver=ns_host, 
)

In [ ]:
master.run(n_iterations=30)

?????**????**?**????**???????:

In [ ]:
from ultraopt import FMinResult

In [ ]:
result = FMinResult(optimizer)
result

??????????`2`?`Worker`, ??**???**????????????`2`

In [ ]:
result.plot_concurrent_over_time(num_points=200);

????????????:

In [ ]:
import pylab as plt

In [ ]:
plt.rcParams['figure.figsize'] = (16, 12)
plt.subplot(2, 2, 1)
result.plot_convergence_over_time();
plt.subplot(2, 2, 2)
ax = result.plot_convergence_over_iter(budget=1, name="budget = 1")
plt.subplot(2, 2, 3)
result.plot_finished_over_time();
plt.subplot(2, 2, 4)
result.plot_correlation_across_budgets();

### ????

?????,????????`Worker`?????,???????????????

?????? `master` , ??master???????????? `worker`

In [ ]:
master.shutdown(shutdown_workers=True)

???? `NameServer`

In [ ]:
NS.shutdown()